# A Simple RAG Demo Project

Before running the notebook, follow the setup instructions in [README.md](README.md).

In [ ]:

# Run once when setting up the project in Colab
#!git clone https://github.com/raivisskadins/simple-rag-demo.git
#%cd simple-rag-demo
#!git pull

# Run once if packages are missing (required also for Colab)
#!pip install -r requirements.txt

# Run to pull latest changes from GitHub
#!git pull


In [ ]:
# ==============================
# 1. Imports
# ==============================

import os
import sys
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from openai import AzureOpenAI
from dotenv import load_dotenv


In [ ]:
# ==============================
# 2. Load environment variables
# ==============================

# Load environment variables
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    os.environ["AZURE_OPENAI_API_KEY"] = userdata.get("AZURE_OPENAI_API_KEY")
    os.environ["AZURE_OPENAI_API_VERSION"] = userdata.get("AZURE_OPENAI_API_VERSION")
    os.environ["AZURE_OPENAI_ENDPOINT"] = userdata.get("AZURE_OPENAI_ENDPOINT")
    os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"] = userdata.get("AZURE_OPENAI_CHAT_DEPLOYMENT")
else:
    load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


In [ ]:
# ==============================
# 3. Load knowledge base
# ==============================

with open("./data/knowledge.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Each paragraph becomes one chunk
chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print("Chunks:")
for c in chunks:
    print("-", c)


In [ ]:
# ==============================
# 4. Create embeddings
# ==============================

def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array(
    [get_embedding(chunk) for chunk in chunks]
).astype("float32")
embedding_dim = chunk_embeddings.shape[1]



# ==============================
# 5. Create FAISS index
# ==============================

index = faiss.IndexFlatIP(embedding_dim)
# remove previously added content
index.reset()
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)


In [ ]:
# ==============================
# 6. User question
# ==============================

#question = "Which ocean is the largest on Earth?"
question = "Is Great Wall of China visible from the Moon?"

print("Question:", question)


# ==============================
# 7. Embed the question
# ==============================

question_embedding = np.array(
    [get_embedding(question)]
).astype("float32")

# You can print out the embedding just in case you want to see how it looks
#print("Question Embedding:", question_embedding)



In [ ]:
# ==============================
# 8. Retrieve top 2 chunks
# ==============================

k = 2

distances, indices = index.search(question_embedding, k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved context:")
for c in retrieved_chunks:
    print("-", c)


In [ ]:
# ==============================
# 9. Create RAG prompt
# ==============================

context = "\n\n".join(retrieved_chunks)

prompt = f"""
# Instructions
Answer the following question in one short response.
Use only the context provided, even if it contains false facts.
Say "There is no information awailable" if there is no answer in the context

# Context:
{context}

# Question:
{question}

# Answer:
"""

#print("Prompt:", prompt)


In [ ]:
# ==============================
# 10. Call GPT-4o to generate the answer
# ==============================

response = client.chat.completions.create(
    model=chat_deployment,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print("\nLLM Answer:")
print(response.choices[0].message.content)